In [19]:
from mpi4py import MPI
import rasterio
import numpy as np
import os
import time

In [2]:
# Initialize MPI
comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

In [11]:
# Load raster
filename = "ortho_gen/CMC_Testpartial_ortho.tiff"  # Input raster file

# Check current working directory
print("Working dir:", os.getcwd())

# Check if file exists
print("File found:", os.path.exists(filename))

Working dir: /project/danperez_1422
File found: True


In [20]:
#Jeho file
# full absolute path
filename = "/project/JehoPark_1421/Student_Files/CMC_Testpartial_ortho.tiff"

print("File found:", os.path.exists(filename))

File found: True


In [21]:
# Load timer
if rank == 0:
    t_start = MPI.Wtime()

In [22]:
# Load raster on root process
if rank == 0:
    t_load_start = MPI.Wtime()
    with rasterio.open(filename) as src:
        data = src.read(1)
        nodata = src.nodata
        shape = data.shape
    t_load_end = MPI.Wtime()
    print(f"[Rank 0] File load time:      {t_load_end - t_load_start:.4f} seconds")
else:
    data = None
    nodata = None
    shape = None


[Rank 0] File load time:      1.1344 seconds


In [23]:
# Broadcast metadata
comm.Barrier()
if rank == 0:
    t_bcast_start = MPI.Wtime()
shape = comm.bcast(shape, root=0)
nodata = comm.bcast(nodata, root=0)

In [15]:
# Scatter data
if rank == 0:
    rows_per_proc = shape[0] // size
    chunks = [data[i*rows_per_proc:(i+1)*rows_per_proc, :] for i in range(size)]
else:
    chunks = None

local_data = comm.scatter(chunks, root=0)

In [16]:
# Compute local mean (ignoring nodata)
local_mean = np.mean(local_data[local_data != nodata])


In [17]:
# Gather and compute global mean
global_mean = comm.reduce(local_mean, op=MPI.SUM, root=0)

In [18]:
if rank == 0:
    global_mean /= size
    print(f"Average Elevation: {global_mean:.2f} meters")

Average Elevation: 116.28 meters


In [24]:
# ── Start timer on rank 0 ──────────────────────────────────────────
if rank == 0:
    t_start = MPI.Wtime()

# Load raster on root process
if rank == 0:
    t_load_start = MPI.Wtime()
    with rasterio.open(filename) as src:
        data = src.read(1)
        nodata = src.nodata
        shape = data.shape
    t_load_end = MPI.Wtime()
    print(f"[Rank 0] File load time:      {t_load_end - t_load_start:.4f} seconds")
else:
    data = None
    nodata = None
    shape = None

# Broadcast metadata
comm.Barrier()
if rank == 0:
    t_bcast_start = MPI.Wtime()

shape = comm.bcast(shape, root=0)
nodata = comm.bcast(nodata, root=0)

comm.Barrier()
if rank == 0:
    print(f"[Rank 0] Broadcast time:      {MPI.Wtime() - t_bcast_start:.4f} seconds")

# Scatter data
if rank == 0:
    t_scatter_start = MPI.Wtime()
    rows_per_proc = shape[0] // size
    chunks = [data[i*rows_per_proc:(i+1)*rows_per_proc, :] for i in range(size)]
else:
    chunks = None

local_data = comm.scatter(chunks, root=0)

comm.Barrier()
if rank == 0:
    print(f"[Rank 0] Scatter time:        {MPI.Wtime() - t_scatter_start:.4f} seconds")

# Compute local mean (ignoring nodata)
t_compute_start = MPI.Wtime()
local_mean = np.mean(local_data[local_data != nodata])
comm.Barrier()
t_compute_end = MPI.Wtime()
if rank == 0:
    print(f"[Rank 0] Compute time:        {t_compute_end - t_compute_start:.4f} seconds")

# Gather and compute global mean
if rank == 0:
    t_reduce_start = MPI.Wtime()

global_mean = comm.reduce(local_mean, op=MPI.SUM, root=0)

if rank == 0:
    print(f"[Rank 0] Reduce time:         {MPI.Wtime() - t_reduce_start:.4f} seconds")

    global_mean /= size
    print(f"\nAverage Elevation:            {global_mean:.2f} meters")

    # ── Total time ────────────────────────────────────────────────────
    t_total = MPI.Wtime() - t_start
    print(f"\n{'='*45}")
    print(f"Total wall time ({size} ranks):   {t_total:.4f} seconds")
    print(f"{'='*45}")

[Rank 0] File load time:      1.1164 seconds
[Rank 0] Broadcast time:      0.0001 seconds
[Rank 0] Scatter time:        0.0261 seconds
[Rank 0] Compute time:        1.5613 seconds
[Rank 0] Reduce time:         0.0002 seconds

Average Elevation:            116.28 meters

Total wall time (1 ranks):   2.7048 seconds
